# Situation #

We have tons of DES SNIa that are needed to be matched up to hosts and we need to upload them to the BLAST server and stuff. So we manuall removed the iaua name containing SNIa (now that we can resort to processing things in BLAST with just using CID's, lets see what happens). 


No need to do anything with TNS server at all. Just living life and stuff. 

### Conceptual reasoning

- **zHEL (heliocentric redshift):**  
  This is the directly measured spectroscopic redshift of the supernova spectrum in the heliocentric frame (corrected only for the Earth’s orbital motion around the Sun). It’s the *raw* transient redshift.

- **zCMB:**  
  That’s the same redshift, but transformed into the frame of the cosmic microwave background dipole (i.e., correcting for our solar system’s motion relative to the CMB). This is useful for **cosmological analysis** (Hubble diagrams), not for identifying or cross-matching a transient.

- **zHD:**  
  That’s a further **Hubble-diagram adjusted redshift**, where peculiar velocity models and flow corrections are folded in. It’s even more “analysis-specific,” not a directly observed value.

---

For BLAST (or any upload that’s just meant to identify the transient itself with its own spectrum and coordinates), you want the **measured, heliocentric redshift**:  
👉 **zHEL is the observational property of the transient.**

Everything else (`zCMB`, `zHD`, host redshifts) are **cosmology tools**, not the transient’s intrinsic spectroscopic measurement.

In [ ]:
import pandas as pd

# point this to your actual file
in_path  = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/DES_no_iaua.csv"

df = pd.read_csv(in_path)

for col in df.columns:
    print(col)


CID
CIDint
IDSURVEY
TYPE
zHEL
zHELERR
zCMB
zCMBERR
zHD
zHDERR
VPEC
VPECERR
MWEBV
HOST_ZSPEC
HOST_ZSPECERR
HOST_RA
HOST_DEC
HOST_ANGSEP
HOST_DDLR
HOST_LOGMASS
HOST_LOGMASS_ERR
HOST_COLOR
HOST_COLOR_ERR
PKMJD
PKMJDERR
x1
x1ERR
c
cERR
mB
mBERR
mB_corr
x0
x0ERR
COV_x1_c
COV_x1_x0
COV_c_x0
NDOF
FITPROB
PROB_SCONE
PROB_SNIRFV19
PROB_SNNDESCC
PROB_SNNJ17
PROB_SNNV19
MU
MUERR_FINAL
PROBCC_BEAMS
biasCor_mu
biasCor_muCOVSCALE
biasCor_muCOVADD


So it turns out that there are no RA and DEC measurements for the SNIa that were part of the DES data release and we must look at the DES .FITS files in order to open stuff up. 

In [ ]:
## here's Some Stuff to get us started:


# === Load LOWZ ===
lowz_path = "DES-SN5YR-1.2/0_DATA/DES-SN5YR_LOWZ/DES-SN5YR_LOWZ_HEAD.FITS.gz"
lowz_table = Table.read(lowz_path, hdu=1).to_pandas()
if lowz_table['SNID'].dtype == object and isinstance(lowz_table['SNID'].iloc[0], bytes):
    lowz_table['SNID'] = lowz_table['SNID'].str.decode('utf-8').str.strip()

# === Load FOUNDATION ===
found_path = "DES-SN5YR-1.2/0_DATA/DES-SN5YR_Foundation/DES-SN5YR_Foundation_HEAD.FITS.gz"
found_table = Table.read(found_path, hdu=1).to_pandas()
if found_table['SNID'].dtype == object and isinstance(found_table['SNID'].iloc[0], bytes):
    found_table['SNID'] = found_table['SNID'].str.decode('utf-8').str.strip()

# === Load Metadata CSV ===
csv_path = "DES5YR_DESI_data/snia_with_missing_host_coords.csv"
meta_df = pd.read_csv(csv_path)
meta_df.rename(columns={"CID": "SNID"}, inplace=True)

# === Merge with LOWZ ===
merged_df = pd.merge(meta_df, lowz_table[['SNID', 'RA', 'DEC']], on='SNID', how='left')

# === Now try to fill missing RA/DEC using FOUNDATION ===
found_coords = found_table[['SNID', 'RA', 'DEC']].copy()
found_coords.rename(columns={'RA': 'RA_FND', 'DEC': 'DEC_FND'}, inplace=True)

# Join foundation coordinates
merged_df = pd.merge(merged_df, found_coords, on='SNID', how='left')

# === Final stitch: prefer RA from LOWZ, fallback to FOUNDATION ===
## fillna fills in the collumn with the argument if the collumn entry 
## is n/a.
merged_df['RA'] = merged_df['RA'].fillna(merged_df['RA_FND'])
merged_df['DEC'] = merged_df['DEC'].fillna(merged_df['DEC_FND'])

# Drop the extra columns
merged_df.drop(columns=['RA_FND', 'DEC_FND'], inplace=True)

# === Save it clean ===
merged_df.to_csv("DES5YR_DESI_data/SNIa_no_hostC_yes_SNIaC_LowZ_FOUNDATION.csv", index=False)

In [ ]:
out_path = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/DES_no_iaua_cleaned_for_BLAST.csv"



# keep only the requested columns
subset = df[["CID", "RA", "DEC", "zHEL", "TYPE"]]

# overwrite TYPE column with "None"
subset["TYPE"] = "None"

# save
subset.to_csv(out_path, index=False)

print("Saved:", out_path)
print(subset.head())

KeyError: "['RA', 'DEC'] not in index"